# RV32I-MLA CPU Micro-Diagnostics  
## Frozen 3-stage CPU bring-up notebook

This notebook is aligned to the **locked 3-stage CPU**:

- **IF** — fetch instruction  
- **ID** — decode and read register operands  
- **EX/MEM/WB** — execute, access memory if needed, and write back  

The current `cpu_top.v` is a compact 3-stage controller with internal wait states for the shared synchronous BRAM interface.

This notebook is for **PYNQ bring-up and fault isolation**:

- verify **PS ↔ BRAM** access
- load a tiny diagnostic program into IMEM
- inspect IMEM / DMEM windows
- localize faults in **store**, **load**, **branch**, and **jump** behavior

---

## Important limitation

This notebook is only **fully trustworthy** if you have a deterministic way to do this exact sequence:

1. hold CPU in reset  
2. clear IMEM / DMEM  
3. write the test program into IMEM  
4. release CPU reset  
5. let the CPU run from `pc = 0`

If your current overlay **does not expose a PS-controlled CPU reset/start**, then staging memory while the CPU is already live can produce **nondeterministic results**.

So this notebook supports two modes:

- **best mode:** stage program, then release/reset CPU cleanly  
- **fallback mode:** stage program and manually use your current board flow, but treat results more cautiously

## 1) Assumed implemented subset

The notebook assumes the current frozen subset:

- **R-type:** `ADD`, `SUB`, `AND`, `OR`
- **I-type:** `ADDI`
- **LOAD:** `LW`
- **STORE:** `SW`
- **BRANCH:** `BEQ`, `BNE`
- **JAL:** PC-relative unconditional jump

It does **not** assume `LUI` or `JALR`.

In [1]:
from pynq import Overlay, MMIO
import time

BIT_NAME = "rv32i_mla_bd.bit"
PREFERRED_MEM_NAME = "axi_bram"   # change only if your exported name differs

## 2) Helper functions

In [2]:
def hr(title):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)

def whex(x):
    return f"0x{(x & 0xFFFFFFFF):08X}"

def mmio_write32(mmio, off, val):
    mmio.write(off, val & 0xFFFFFFFF)

def mmio_read32(mmio, off):
    return mmio.read(off) & 0xFFFFFFFF

def clear_window(mmio, base, size_bytes):
    for off in range(0, size_bytes, 4):
        mmio_write32(mmio, base + off, 0)

def load_program(mmio, words, base=0x0000, verbose=True):
    for i, w in enumerate(words):
        mmio_write32(mmio, base + 4 * i, w)
        if verbose:
            print(f"IMEM[{i:02d}] @ {whex(base + 4*i)} <= {whex(w)}")

def dump_words(mmio, base, count, label="MEM"):
    for i in range(count):
        addr = base + 4 * i
        print(f"{label}[{i:02d}] @ {whex(addr)} = {whex(mmio_read32(mmio, addr))}")

def select_bram_mmio(overlay, preferred_name=PREFERRED_MEM_NAME):
    if preferred_name in overlay.mem_dict:
        info = overlay.mem_dict[preferred_name]
        base = info["phys_addr"]
        size = info["addr_range"]
        print(f"Using preferred BRAM region: {preferred_name}")
        return MMIO(base, size), preferred_name, base, size

    if len(overlay.mem_dict) == 0:
        raise RuntimeError("No memory regions found in overlay.mem_dict")

    name = list(overlay.mem_dict.keys())[0]
    info = overlay.mem_dict[name]
    base = info["phys_addr"]
    size = info["addr_range"]
    print(f"Preferred BRAM region not found; using fallback: {name}")
    return MMIO(base, size), name, base, size

def print_result_summary(mmio):
    d0 = mmio_read32(mmio, DMEM_BASE + 0x00)
    d1 = mmio_read32(mmio, DMEM_BASE + 0x04)
    print(f"DMEM[0] @ {whex(DMEM_BASE + 0x00)} = {whex(d0)} ({d0})")
    print(f"DMEM[1] @ {whex(DMEM_BASE + 0x04)} = {whex(d1)} ({d1})")
    return d0, d1

def stage_program(mmio, prog, clear_dmem=True, clear_reserved_mmio=True, verbose=True):
    clear_window(mmio, IMEM_BASE, 0x80)
    if clear_dmem:
        clear_window(mmio, DMEM_BASE, 0x40)
    if clear_reserved_mmio:
        clear_window(mmio, MMIO_BASE, 0x10)
    load_program(mmio, prog, IMEM_BASE, verbose=verbose)
    time.sleep(0.01)

def show_windows(mmio):
    hr("IMEM dump")
    dump_words(mmio, IMEM_BASE, 16, "IMEM")
    hr("DMEM dump")
    dump_words(mmio, DMEM_BASE, 8, "DMEM")
    hr("Reserved MMIO window dump")
    dump_words(mmio, MMIO_BASE, 4, "MMIO")

## 3) Frozen CPU-visible memory map

This notebook uses the same software-side convention:

- `IMEM_BASE = 0x0000`
- `DMEM_BASE = 0x0100`
- `MMIO_BASE = 0x0200`

Right now, the `0x0200+` area is treated as **reserved/debug space**, not as a guaranteed implemented start/done MMIO block.

In [3]:
IMEM_BASE   = 0x0000
DMEM_BASE   = 0x0100
MMIO_BASE   = 0x0200

MMIO_START  = 0x0200   # reserved / future use
MMIO_DONE   = 0x0204   # reserved / future use
MMIO_CYCLES = 0x0208   # reserved / future use

print("IMEM_BASE   =", whex(IMEM_BASE))
print("DMEM_BASE   =", whex(DMEM_BASE))
print("MMIO_BASE   =", whex(MMIO_BASE))
print("MMIO_START  =", whex(MMIO_START))
print("MMIO_DONE   =", whex(MMIO_DONE))
print("MMIO_CYCLES =", whex(MMIO_CYCLES))

IMEM_BASE   = 0x00000000
DMEM_BASE   = 0x00000100
MMIO_BASE   = 0x00000200
MMIO_START  = 0x00000200
MMIO_DONE   = 0x00000204
MMIO_CYCLES = 0x00000208


## 4) Encoding helpers

In [4]:
def enc_r(funct7, rs2, rs1, funct3, rd, opcode=0b0110011):
    return ((funct7 & 0x7F) << 25) | ((rs2 & 0x1F) << 20) | ((rs1 & 0x1F) << 15) | ((funct3 & 0x7) << 12) | ((rd & 0x1F) << 7) | (opcode & 0x7F)

def enc_i(imm, rs1, funct3, rd, opcode):
    imm &= 0xFFF
    return (imm << 20) | ((rs1 & 0x1F) << 15) | ((funct3 & 0x7) << 12) | ((rd & 0x1F) << 7) | (opcode & 0x7F)

def enc_s(imm, rs2, rs1, funct3, opcode=0b0100011):
    imm &= 0xFFF
    imm_11_5 = (imm >> 5) & 0x7F
    imm_4_0  = imm & 0x1F
    return (imm_11_5 << 25) | ((rs2 & 0x1F) << 20) | ((rs1 & 0x1F) << 15) | ((funct3 & 0x7) << 12) | (imm_4_0 << 7) | (opcode & 0x7F)

def enc_b(imm, rs2, rs1, funct3, opcode=0b1100011):
    imm &= 0x1FFF
    bit12    = (imm >> 12) & 0x1
    bit11    = (imm >> 11) & 0x1
    bits10_5 = (imm >> 5) & 0x3F
    bits4_1  = (imm >> 1) & 0xF
    return (bit12 << 31) | (bits10_5 << 25) | ((rs2 & 0x1F) << 20) | ((rs1 & 0x1F) << 15) | ((funct3 & 0x7) << 12) | (bits4_1 << 8) | (bit11 << 7) | (opcode & 0x7F)

def enc_j(imm, rd, opcode=0b1101111):
    imm &= 0x1FFFFF
    bit20     = (imm >> 20) & 0x1
    bits10_1  = (imm >> 1) & 0x3FF
    bit11     = (imm >> 11) & 0x1
    bits19_12 = (imm >> 12) & 0xFF
    return (bit20 << 31) | (bits19_12 << 12) | (bit11 << 20) | (bits10_1 << 21) | ((rd & 0x1F) << 7) | (opcode & 0x7F)

def rv_add(rd, rs1, rs2):   return enc_r(0b0000000, rs2, rs1, 0b000, rd)
def rv_sub(rd, rs1, rs2):   return enc_r(0b0100000, rs2, rs1, 0b000, rd)
def rv_and(rd, rs1, rs2):   return enc_r(0b0000000, rs2, rs1, 0b111, rd)
def rv_or(rd, rs1, rs2):    return enc_r(0b0000000, rs2, rs1, 0b110, rd)
def rv_addi(rd, rs1, imm):  return enc_i(imm, rs1, 0b000, rd, 0b0010011)
def rv_lw(rd, rs1, imm):    return enc_i(imm, rs1, 0b010, rd, 0b0000011)
def rv_sw(rs2, rs1, imm):   return enc_s(imm, rs2, rs1, 0b010, 0b0100011)
def rv_beq(rs1, rs2, imm):  return enc_b(imm, rs2, rs1, 0b000, 0b1100011)
def rv_bne(rs1, rs2, imm):  return enc_b(imm, rs2, rs1, 0b001, 0b1100011)
def rv_jal(rd, imm):        return enc_j(imm, rd, 0b1101111)

## 5) Diagnostic programs

Recommended progression:

1. **SMOKE** — absolute minimum: execute anything and store one result  
2. **NO_BRANCH** — final store path without branch or jump  
3. **BRANCH_ONLY** — verifies load + compare + branch target  
4. **JUMP_ONLY** — verifies JAL target handling  

Expected pass markers:

- `DMEM[0x0100] = 12`
- `DMEM[0x0104] = 2`

The **SMOKE** test checks only `DMEM[0x0100]`.

In [5]:
prog_smoke = [
    rv_addi(1, 0, 12),
    rv_sw(1, 0, 0x100),
    rv_jal(0, 0),
]

prog_no_branch = [
    rv_addi(1, 0, 5),
    rv_addi(2, 0, 7),
    rv_add(3, 1, 2),
    rv_sw(3, 0, 0x100),
    rv_addi(5, 0, 2),
    rv_sw(5, 0, 0x104),
    rv_jal(0, 0),
]

prog_branch_only = [
    rv_addi(1, 0, 5),
    rv_addi(2, 0, 7),
    rv_add(3, 1, 2),
    rv_sw(3, 0, 0x100),
    rv_lw(4, 0, 0x100),
    rv_beq(3, 4, 8),
    rv_addi(5, 0, 1),
    rv_addi(5, 0, 2),
    rv_sw(5, 0, 0x104),
    rv_jal(0, 0),
]

prog_jump_only = [
    rv_addi(1, 0, 5),
    rv_addi(2, 0, 7),
    rv_add(3, 1, 2),
    rv_sw(3, 0, 0x100),
    rv_jal(0, 8),
    rv_addi(5, 0, 1),
    rv_addi(5, 0, 2),
    rv_sw(5, 0, 0x104),
    rv_jal(0, 0),
]

TESTS = {
    "SMOKE": {
        "program": prog_smoke,
        "expect_dmem0": 12,
        "expect_dmem1": None,
        "note": "Minimal execute/store sanity check",
    },
    "NO_BRANCH": {
        "program": prog_no_branch,
        "expect_dmem0": 12,
        "expect_dmem1": 2,
        "note": "Final store path without branch/jump",
    },
    "BRANCH_ONLY": {
        "program": prog_branch_only,
        "expect_dmem0": 12,
        "expect_dmem1": 2,
        "note": "BEQ compare + target handling",
    },
    "JUMP_ONLY": {
        "program": prog_jump_only,
        "expect_dmem0": 12,
        "expect_dmem1": 2,
        "note": "JAL target handling",
    },
}

for name, meta in TESTS.items():
    print(f"{name:12s} : {meta['note']}")

SMOKE        : Minimal execute/store sanity check
NO_BRANCH    : Final store path without branch/jump
BRANCH_ONLY  : BEQ compare + target handling
JUMP_ONLY    : JAL target handling


## 6) Load overlay and create BRAM MMIO handle

In [6]:
hr("Load Overlay")
ol = Overlay(BIT_NAME)

print(f"Loaded bitstream : {BIT_NAME}")
print(f"IP count         : {len(ol.ip_dict)}")
print(f"Memory regions   : {len(ol.mem_dict)}")

hr("overlay.mem_dict")
for name, info in ol.mem_dict.items():
    print(name, "=>", info)

mmio, mem_name, mem_base, mem_size = select_bram_mmio(ol, PREFERRED_MEM_NAME)

hr("Selected BRAM MMIO")
print("Name :", mem_name)
print("Base :", whex(mem_base))
print("Size :", whex(mem_size))


Load Overlay
Loaded bitstream : rv32i_mla_bd.bit
IP count         : 1
Memory regions   : 2

overlay.mem_dict
axi_bram => {'fullpath': 'axi_bram', 'type': 'DDR4', 'bdtype': None, 'state': None, 'addr_range': 8192, 'phys_addr': 1073741824, 'mem_id': 'S_AXI', 'memtype': 'MEMORY', 'gpio': {}, 'interrupts': {}, 'parameters': {'C_BRAM_INST_MODE': 'EXTERNAL', 'C_MEMORY_DEPTH': '2048', 'C_BRAM_ADDR_WIDTH': '11', 'C_S_AXI_ADDR_WIDTH': '13', 'C_S_AXI_DATA_WIDTH': '32', 'C_S_AXI_ID_WIDTH': '12', 'C_S_AXI_PROTOCOL': 'AXI4', 'C_S_AXI_SUPPORTS_NARROW_BURST': '0', 'C_SINGLE_PORT_BRAM': '1', 'C_FAMILY': 'zynq', 'C_READ_LATENCY': '1', 'C_RD_CMD_OPTIMIZATION': '0', 'C_S_AXI_CTRL_ADDR_WIDTH': '32', 'C_S_AXI_CTRL_DATA_WIDTH': '32', 'C_ECC': '0', 'C_ECC_TYPE': '0', 'C_FAULT_INJECT': '0', 'C_ECC_ONOFF_RESET_VALUE': '0', 'DATA_WIDTH': '32', 'ID_WIDTH': '12', 'PROTOCOL': 'AXI4', 'SUPPORTS_NARROW_BURST': '0', 'SINGLE_PORT_BRAM': '1', 'ECC_TYPE': '0', 'USE_ECC': '0', 'FAULT_INJECT': '0', 'ECC_ONOFF_RESET_VALUE

## 7) PS ↔ BRAM sanity check

Run this before any CPU test.

If plain PS read/write fails here, stop and fix the memory mapping first.

In [7]:
hr("PS Write/Read Sanity")

tests = [
    (0x00, 0xAAAAAAAA),
    (0x04, 0x55555555),
    (0x08, 0x12345678),
    (0x0C, 0xDEADBEEF),
]

for off, val in tests:
    before = mmio_read32(mmio, off)
    mmio_write32(mmio, off, val)
    after = mmio_read32(mmio, off)
    print(f"off={whex(off)} before={whex(before)} wrote={whex(val)} after={whex(after)}")

print("\nIf AFTER does not match WROTE, do not continue.")


PS Write/Read Sanity
off=0x00000000 before=0x00000000 wrote=0xAAAAAAAA after=0xAAAAAAAA
off=0x00000004 before=0x00000000 wrote=0x55555555 after=0x55555555
off=0x00000008 before=0x00000000 wrote=0x12345678 after=0x12345678
off=0x0000000C before=0x00000000 wrote=0xDEADBEEF after=0xDEADBEEF

If AFTER does not match WROTE, do not continue.


## 8) Select and stage a diagnostic program

Set `SELECTED_TEST` to one of:

- `"SMOKE"`
- `"NO_BRANCH"`
- `"BRANCH_ONLY"`
- `"JUMP_ONLY"`

This cell stages memory only. It does **not** itself guarantee a clean CPU restart.

In [8]:
SELECTED_TEST = "SMOKE"

meta = TESTS[SELECTED_TEST]
prog = meta["program"]

hr(f"Stage program: {SELECTED_TEST}")
print("Purpose :", meta["note"])
print("Expected DMEM[0x0100] =", meta["expect_dmem0"])
print("Expected DMEM[0x0104] =", meta["expect_dmem1"])

stage_program(mmio, prog, clear_dmem=True, clear_reserved_mmio=True, verbose=True)

hr("Program words now in IMEM")
dump_words(mmio, IMEM_BASE, len(prog), "IMEM")


Stage program: SMOKE
Purpose : Minimal execute/store sanity check
Expected DMEM[0x0100] = 12
Expected DMEM[0x0104] = None
IMEM[00] @ 0x00000000 <= 0x00C00093
IMEM[01] @ 0x00000004 <= 0x10102023
IMEM[02] @ 0x00000008 <= 0x0000006F

Program words now in IMEM
IMEM[00] @ 0x00000000 = 0x00C00093
IMEM[01] @ 0x00000004 = 0x10102023
IMEM[02] @ 0x00000008 = 0x0000006F


## 9) Deterministic run control

### Preferred
Use a **real PS-controlled CPU reset/start mechanism** so you can:

- stage program
- release reset
- wait briefly
- read results

### Fallback
If your current bitstream does **not** expose reset/start, use your current manual board flow here. But interpret results with caution, because the CPU may already have been live while you were writing IMEM.

### Suggested manual pause
After your run action, wait a short time before reading DMEM.

In [9]:
RUN_WAIT_SEC = 0.05
print(f"After your CPU run/reset action, wait about {RUN_WAIT_SEC:.3f} seconds before reading DMEM.")
time.sleep(RUN_WAIT_SEC)
print("Ready to read results.")

After your CPU run/reset action, wait about 0.050 seconds before reading DMEM.
Ready to read results.


## 10) Read results

In [10]:
hr(f"Results for {SELECTED_TEST}")

d0, d1 = print_result_summary(mmio)

exp0 = meta["expect_dmem0"]
exp1 = meta["expect_dmem1"]

ok0 = (exp0 is None) or (d0 == exp0)
ok1 = (exp1 is None) or (d1 == exp1)

print("\nCheck summary:")
print("DMEM[0x0100] :", "PASS" if ok0 else "FAIL")
print("DMEM[0x0104] :", "PASS" if ok1 else "FAIL")

if ok0 and ok1:
    print("\nOverall     : PASS")
else:
    print("\nOverall     : FAIL / NEEDS DEBUG")


Results for SMOKE
DMEM[0] @ 0x00000100 = 0x0000000C (12)
DMEM[1] @ 0x00000104 = 0x00000000 (0)

Check summary:
DMEM[0x0100] : PASS
DMEM[0x0104] : PASS

Overall     : PASS


## 11) Interpretation

### SMOKE
- `DMEM[0x0100] = 12`  
  instruction fetch, decode, execute, and store path are alive at a minimal level

### NO_BRANCH
- `0x0100 = 12, 0x0104 = 2`  
  final store path is good without branch/jump
- `0x0100 = 12, 0x0104 != 2`  
  likely issue in later instruction sequencing, second writeback, or final store path

### BRANCH_ONLY
- `0x0104 = 2`  
  BEQ decision and target are good
- `0x0104 = 1`  
  branch not taken when it should have been
- `0x0104 = 0`  
  later control flow or final store did not complete

### JUMP_ONLY
- `0x0104 = 2`  
  JAL handling is good
- `0x0104 = 1`  
  jump target or skip is wrong
- `0x0104 = 0`  
  later sequencing or final store did not complete

---

## Very useful localization rule

If **NO_BRANCH already fails**, then the problem is **not branch logic**.  
That means the bug is likely in:

- later straight-line instruction sequencing
- second result register writeback
- second store issue to `0x0104`
- PC progression after the first store

## 12) Debug dump

In [11]:
show_windows(mmio)


IMEM dump
IMEM[00] @ 0x00000000 = 0x00C00093
IMEM[01] @ 0x00000004 = 0x10102023
IMEM[02] @ 0x00000008 = 0x0000006F
IMEM[03] @ 0x0000000C = 0x00000000
IMEM[04] @ 0x00000010 = 0x00000000
IMEM[05] @ 0x00000014 = 0x00000000
IMEM[06] @ 0x00000018 = 0x00000000
IMEM[07] @ 0x0000001C = 0x00000000
IMEM[08] @ 0x00000020 = 0x00000000
IMEM[09] @ 0x00000024 = 0x00000000
IMEM[10] @ 0x00000028 = 0x00000000
IMEM[11] @ 0x0000002C = 0x00000000
IMEM[12] @ 0x00000030 = 0x00000000
IMEM[13] @ 0x00000034 = 0x00000000
IMEM[14] @ 0x00000038 = 0x00000000
IMEM[15] @ 0x0000003C = 0x00000000

DMEM dump
DMEM[00] @ 0x00000100 = 0x0000000C
DMEM[01] @ 0x00000104 = 0x00000000
DMEM[02] @ 0x00000108 = 0x00000000
DMEM[03] @ 0x0000010C = 0x00000000
DMEM[04] @ 0x00000110 = 0x00000000
DMEM[05] @ 0x00000114 = 0x00000000
DMEM[06] @ 0x00000118 = 0x00000000
DMEM[07] @ 0x0000011C = 0x00000000

Reserved MMIO window dump
MMIO[00] @ 0x00000200 = 0x00000000
MMIO[01] @ 0x00000204 = 0x00000000
MMIO[02] @ 0x00000208 = 0x00000000
MMIO[0